<a href="https://colab.research.google.com/github/YinlinZhong820/StockPrediction/blob/main/stock_predictio_train_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 14.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from __future__ import division, print_function, unicode_literals,  absolute_import

%matplotlib inline

from google.colab import files

# Common imports
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import optuna
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, accuracy_score
import gc
from sklearn.preprocessing import StandardScaler

use_cuda = True
device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")

In [4]:
device

device(type='cuda')

In [5]:
df = pd.read_csv("drive/MyDrive/data/train.csv")

df.head()

,ID,05/04/2010,06/04/2010,07/04/2010,08/04/2010,09/04/2010,12/04/2010,13/04/2010,14/04/2010,15/04/2010,...,18/03/2022,21/03/2022,22/03/2022,23/03/2022,24/03/2022,25/03/2022,28/03/2022,29/03/2022,30/03/2022,31/03/2022
0,company_0,1.30,0.19,0.46,-0.23,1.20,-0.26,-0.34,1.64,1.21,...,-0.36,0.05,0.30,-3.10,0.04,0.86,0.31,0.95,0.68,-0.52
1,company_1,-0.83,0.62,-2.74,6.33,1.32,4.52,0.05,3.94,5.01,...,2.20,-3.67,3.20,-1.61,2.92,0.93,1.55,3.96,0.41,1.08
2,company_2,-0.02,0.13,-0.70,0.23,1.50,-0.23,1.35,2.26,0.37,...,-0.16,-0.74,2.52,-2.06,1.57,0.10,0.90,3.05,-1.24,-1.26
3,company_3,-1.10,-0.73,-2.98,1.98,3.00,1.01,3.60,4.30,1.05,...,2.47,-2.41,1.10,-4.08,2.09,2.02,2.74,2.33,0.80,-0.91
4,company_4,2.23,-1.45,-1.74,-1.13,0.80,1.29,0.62,3.06,-0.73,...,1.28,-1.96,0.19,-2.28,-1.13,-1.53,-0.64,2.32,-1.64,-4.58


In [6]:
print('shape',df.shape)
print('columns',df.columns.tolist())

# keep only numeric columns (daily percentage columns)
num = df.select_dtypes(include="number")

# flatten and count
positives = (num > 0).sum().sum()
negatives = (num < 0).sum().sum()
zeros = (num == 0).sum().sum()

print(f"positive values: {positives}")
print(f"negative values: {negatives}")
print(f"zeros: {zeros}")
print(f"total numeric entries: {num.size}")

shape (442, 3022)
columns ['ID', '05/04/2010', '06/04/2010', '07/04/2010', '08/04/2010', '09/04/2010', '12/04/2010', '13/04/2010', '14/04/2010', '15/04/2010', '16/04/2010', '19/04/2010', '20/04/2010', '21/04/2010', '22/04/2010', '23/04/2010', '26/04/2010', '27/04/2010', '28/04/2010', '29/04/2010', '30/04/2010', '03/05/2010', '04/05/2010', '05/05/2010', '06/05/2010', '07/05/2010', '10/05/2010', '11/05/2010', '12/05/2010', '13/05/2010', '14/05/2010', '17/05/2010', '18/05/2010', '19/05/2010', '20/05/2010', '21/05/2010', '24/05/2010', '25/05/2010', '26/05/2010', '27/05/2010', '28/05/2010', '01/06/2010', '02/06/2010', '03/06/2010', '04/06/2010', '07/06/2010', '08/06/2010', '09/06/2010', '10/06/2010', '11/06/2010', '14/06/2010', '15/06/2010', '16/06/2010', '17/06/2010', '18/06/2010', '21/06/2010', '22/06/2010', '23/06/2010', '24/06/2010', '25/06/2010', '28/06/2010', '29/06/2010', '30/06/2010', '01/07/2010', '02/07/2010', '06/07/2010', '07/07/2010', '08/07/2010', '09/07/2010', '12/07/2010', '

In [7]:
data_array = df.iloc[:, 1:].values
feature_dim =10

In [8]:
class StockRNN(nn.Module):
    def __init__(self, num_stocks, embedding_dim, hidden_dim, seq_len, num_layers,
                 dropout, learning_rate, weight_decay):
        super(StockRNN, self).__init__()
        self.num_layers = num_layers
        self.hidden_size =  hidden_dim
        self.embedding = nn.Embedding(num_stocks, embedding_dim)
        self.rnn = nn.GRU(input_size=feature_dim + embedding_dim, hidden_size=hidden_dim,
                          num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

        self.criterion = nn.HuberLoss(delta=1.0)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate, weight_decay=weight_decay)

    def forward(self, x_seq, stock_id):
        embbeding = self.embedding(stock_id)
        embbeding = embbeding.unsqueeze(1).repeat(1, x_seq.size(1), 1)
        x = torch.cat([x_seq, embbeding], dim=-1)
        rnn_out, _ = self.rnn(x)
        pooled = rnn_out.mean(dim=1)
        last_rnn_out = self.dropout(pooled)

        logits = self.fc(last_rnn_out).squeeze(1)
        return logits

    def train_one_epoch(self, dataloader):
        self.train()
        total_loss = 0
        for X, y, stocks in dataloader:
            X, stocks = X.to(device), stocks.to(device)
            y = y.to(device).float()
            logits = self(X, stocks)
            loss = self.criterion(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.rnn.parameters(), 1.0)
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(dataloader)

    def predict(self, dataloader):
        self.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for X, y, stocks in dataloader:
                X, stocks = X.to(device), stocks.to(device)
                y = y.to(device).float()
                predicted = self(X, stocks)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
        return all_labels, all_preds

In [9]:
class StockDataset(Dataset):
    def __init__(self, data_tensor, window_size=10):
        self.window_size = window_size
        self.data = data_tensor
        self.num_stocks = data_tensor.shape[0]
        self.total_days = data_tensor.shape[1]
        self.market_return = self.data.mean(dim=0)
        self.features = self.build_features(data_tensor)
        self.valid_samples_per_stock = self.total_days - window_size - 5

    def __len__(self):
        return self.num_stocks * self.valid_samples_per_stock

    def __getitem__(self, idx):
        stock_idx = idx // self.valid_samples_per_stock
        day_offset = idx % self.valid_samples_per_stock

        x_seq = self.features[
            stock_idx,
            day_offset : day_offset + self.window_size
        ]

        y = self.data[stock_idx, day_offset + self.window_size] * 100

        return x_seq, y, torch.tensor(stock_idx, dtype=torch.long)

    def _get_rolling(self, x, window):
        padded = F.pad(x.unsqueeze(1), (window - 1, 0))
        return F.avg_pool1d(padded, kernel_size=window, stride=1).squeeze(1)

    def build_features(self, data):
        num_stocks, num_days = data.shape
        returns = data

        mean_5 = self._get_rolling(data, 5)
        mean_sq_5 = self._get_rolling(data**2, 5)
        std_5 = torch.sqrt(torch.clamp(mean_sq_5 - mean_5**2, min=1e-9))

        mean_20 = self._get_rolling(data, 20)
        mean_sq_20 = self._get_rolling(data**2, 20)
        std_20 = torch.sqrt(torch.clamp(mean_sq_20 - mean_20**2, min=1e-9))

        mom_5 = data - torch.roll(data, shifts=5, dims=1)
        mom_5[:, :5] = 0

        mom_20 = data - torch.roll(data, shifts=20, dims=1)
        mom_20[:, :20] = 0

        relative_str = data - self.market_return

        padded_for_unfold = F.pad(data, (4, 0))
        windows = padded_for_unfold.unfold(1, 5, 1) # [num_stocks, num_days, 5]
        max_5, _ = windows.max(dim=-1)
        min_5, _ = windows.min(dim=-1)

        for i in range(data.shape[1]):
            max_5[:, i] = data[:, max(0, i-4):i+1].max(dim=1).values
            min_5[:, i] = data[:, max(0, i-4):i+1].min(dim=1).values

        features = torch.stack([
            returns,        # 1. 原始收益
            mean_5,         # 2. 短期均值
            std_5,          # 3. 短期波动
            mom_5,          # 4. 短期动量
            max_5,          # 5. 短期高点
            min_5,          # 6. 短期低点
            mean_20,        # 7. 长期均值 (NEW)
            std_20,         # 8. 长期波动 (NEW)
            mom_20,         # 9. 长期动量 (NEW)
            relative_str    # 10. 相对大盘强度 (NEW)
        ], dim=-1)

        eps = 1e-8
        f_mean = features.mean(dim=(0, 1), keepdim=True)
        f_std = features.std(dim=(0, 1), keepdim=True)

        features = (features - f_mean) / (f_std + eps)

        return features


In [10]:
num_stocks, total_days = data_array.shape
split_idx = int(total_days * 0.9)

train_data_raw = data_array[:, :split_idx]
test_data_raw = data_array[:, split_idx:]


scaler = StandardScaler()
train_data_scaled = scaler.fit_transform(train_data_raw.T).T.astype(np.float32)
test_data_scaled = scaler.transform(test_data_raw.T).T.astype(np.float32)

train_tensor = torch.from_numpy(train_data_scaled)
test_tensor = torch.from_numpy(test_data_scaled).float()

print(f"Training Set has: {train_data_raw.shape[1]} days, Test Set has: {test_data_raw.shape[1]} days")

Training Set has: 2718 days, Test Set has: 303 days


In [11]:
study = optuna.load_study(
    study_name="stock_prediction",
    storage="sqlite:////content/drive/MyDrive/data/params_3_20_4.db"
)
best_params = study.best_params
print(best_params)

{'lr': 0.0008917998542781052, 'hidden_size': 16, 'emb_dim': 8, 'win_size': 45, 'dropout': 0.7296423121975836, 'num_layers': 1, 'weight_decay': 0.00011340169105363128}


In [27]:
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0, path='checkpoint'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_acc = None
        self.best_loss = None
        self.early_stop = False
        self.delta = delta
        self.path = path

        if not os.path.exists(self.path):
            os.makedirs(self.path)

    def __call__(self, val_loss, val_ic, model):

        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_ic = val_ic
            self.save_checkpoint(val_loss, val_ic, model, "init")

        elif val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, val_ic, model, "best_loss")
            self.counter = 0

        elif val_ic > self.best_ic:
            self.best_ic = val_ic
            self.save_checkpoint(val_loss, val_ic, model, "best_ic")

        else:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_loss, val_ic, model, tag):
        '''
        tag: 'best_loss' 或 'best_ic'
        '''
        suffix = val_loss if tag == 'best_loss' else val_ic
        file_path = os.path.join(self.path, f"/content/drive/MyDrive/data/3-20-run1/{tag}_model_{suffix}.pt")
        if self.verbose:
            print(f'[{tag.upper()}] New record! Loss: {val_loss:.4f}, IC: {val_ic:.4f}. Saving...')

        torch.save(model.state_dict(), file_path)

In [28]:
best_params = {
    'lr': 0.0002928065808927338,
    'hidden_size': 64,
    'emb_dim': 32,
    'win_size': 45,
    'dropout': 0.6298966978855414,
    'num_layers': 2,
    'weight_decay': 0.00014677762132168205
}



EPOCHS = 50
PATIENCE = 10

######### Full Training ############
early_stopping = EarlyStopping(patience=PATIENCE, verbose=True)

train_ds = StockDataset(train_tensor, window_size=best_params['win_size'])
test_ds = StockDataset(test_tensor, window_size=best_params['win_size'])

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, pin_memory=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=512, pin_memory=True, num_workers=2)


model = StockRNN(
    num_stocks=442, embedding_dim=best_params['emb_dim'],
    hidden_dim=best_params['hidden_size'], weight_decay=best_params['weight_decay'],
    seq_len=best_params['win_size'], dropout=best_params["dropout"], learning_rate=best_params['lr'],
    num_layers=best_params['num_layers']).to(device)

# checkpoint = torch.load('/content/drive/MyDrive/data/best_loss_model_3_19.pt')
# model.load_state_dict(checkpoint)

best_acc = 0.0
model.to(device)
for epoch in range(EPOCHS):
    model.train()
    avg_loss = model.train_one_epoch(train_loader)

    y_pred = []
    y_true = []
    probs = []
    best_val_loss = 0
    val_loss = 0
    model.eval()
    with torch.no_grad():
        for X, y, stocks in test_loader:
            X, stocks = X.to(device), stocks.to(device)
            y = y.to(device).float()
            predicted = model(X, stocks)
            val_loss += model.criterion(predicted, y).item()

            y_pred.extend(predicted.cpu().numpy())
            y_true.extend(y.cpu().numpy())

    val_avg_loss = val_loss / len(test_loader)
    acc = accuracy_score(y_true, np.round(y_pred))

    ic, _ = stats.pearsonr(y_pred, y_true)

    current_lr = model.optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {(avg_loss):.4f} | Val Loss: {(val_avg_loss):.4f} | Val ic: {ic:.4f} | Current LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'model_lowest_loss.pt')

    # model.scheduler.step()

    early_stopping(val_avg_loss, ic, model)

    if early_stopping.early_stop:
        print("Early stopping triggered. Training stopped.")
        break


del model, train_loader, test_loader, train_ds, test_ds
torch.cuda.empty_cache()
gc.collect()

print(f"Best val loss: {val_avg_loss:.4f}")

Epoch [1/50] | Train Loss: 65.1147 | Val Loss: 67.9137 | Val ic: 0.0129 | Current LR: 0.000293
[INIT] New record! Loss: 67.9137, IC: 0.0129. Saving...
Epoch [2/50] | Train Loss: 65.0996 | Val Loss: 67.9146 | Val ic: 0.0124 | Current LR: 0.000293
EarlyStopping counter: 1 out of 10
Epoch [3/50] | Train Loss: 65.0853 | Val Loss: 67.9199 | Val ic: 0.0076 | Current LR: 0.000293
EarlyStopping counter: 2 out of 10
Epoch [4/50] | Train Loss: 65.0655 | Val Loss: 67.9176 | Val ic: 0.0090 | Current LR: 0.000293
EarlyStopping counter: 3 out of 10
Epoch [5/50] | Train Loss: 65.0441 | Val Loss: 67.9191 | Val ic: 0.0142 | Current LR: 0.000293
[BEST_IC] New record! Loss: 67.9191, IC: 0.0142. Saving...
Epoch [6/50] | Train Loss: 65.0114 | Val Loss: 67.9190 | Val ic: 0.0163 | Current LR: 0.000293
[BEST_IC] New record! Loss: 67.9190, IC: 0.0163. Saving...
Epoch [7/50] | Train Loss: 64.9783 | Val Loss: 67.9062 | Val ic: 0.0225 | Current LR: 0.000293
[BEST_LOSS] New record! Loss: 67.9062, IC: 0.0225. Savin

KeyboardInterrupt: 

In [29]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # 清理多进程留下的缓存

In [60]:
def load_model(model_path):
    model = StockRNN(
        num_stocks=442, embedding_dim=best_params['emb_dim'],
        hidden_dim=best_params['hidden_size'], weight_decay=best_params['weight_decay'],
        seq_len=best_params['win_size'], dropout=best_params["dropout"], learning_rate=best_params['lr'],
        num_layers=best_params['num_layers']).to(device)

    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

def get_raw_prediction(model):
    y_pred_raw = []
    with torch.no_grad():
        for i in range(442):
            last_idx = (i * samples_per_stock) + (samples_per_stock - 1)
            X, _, stock_idx = predict_ds[last_idx]
            X = X.unsqueeze(0).to(device)
            stock_idx = torch.tensor([stock_idx]).to(device)

            predicted = model(X, stock_idx)
            y_pred_raw.extend(predicted.cpu().numpy())

    return np.array(y_pred_raw)


In [61]:
#################Prediction#############################
from collections import Counter


buffer=60
win_size = best_params["win_size"]
recent_data = data_array[:, -(win_size + buffer):]

scaler = StandardScaler()
recent_data_scaled = scaler.fit_transform(recent_data.T).T
recent_data_tensor=  torch.from_numpy(recent_data_scaled).float()
predict_ds = StockDataset(recent_data_tensor, window_size=win_size)
samples_per_stock = predict_ds.valid_samples_per_stock

model_ic = load_model('/content/drive/MyDrive/data/best_model/best_ic_model_0.0197.pt')

print("Getting predictions from both models...")
y_pred_array = get_raw_prediction(model_ic) / 100.0
y_pred_centered = y_pred_array - y_pred_array.mean()

k = 40
y_probs = 1 / (1 + np.exp(-k * y_pred_centered))

submission = pd.DataFrame({
    'ID': [f'company_{i}' for i in range(442)],
    'value': np.clip(y_probs, 0.25, 0.75)
})

submission.to_csv('/content/drive/MyDrive/data/best_model/final_submission_prob_emsemble.csv', index=False)
print("submission.csv generated")


Getting predictions from both models...
submission.csv generated


In [42]:
# --- 检查结果 ---
print("\nFinal Statistics:")
print(f"Mean Probability: {submission['value'].mean():.4f} (Ideal: ~0.5)")
print(f"Max Probability:  {submission['value'].max():.4f}")
print(f"Min Probability:  {submission['value'].min():.4f}")

# 看看大于 0.5 (看涨) 的股票占比
pos_count = (submission['value'] > 0.5).sum()
print(f"Bullish Predictions (>0.5): {pos_count} out of 442")


Final Statistics:
Mean Probability: 0.4807 (Ideal: ~0.5)
Max Probability:  0.9977
Min Probability:  0.0064
Bullish Predictions (>0.5): 188 out of 442
